## 1. Load & Profile

In [1]:
import pandas as pd

df = pd.read_csv("sales_messy.csv")
customers = pd.read_csv("customers.csv")

In [2]:
print("Shape:", df.shape)

df.info()

print("Missing values:")
print(df.isnull().sum())

print("Duplicates:", df.duplicated().sum())

print("countries:")
print(df["country"].unique())

Shape: (208, 9)
<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB
Missing values:
order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64
Duplicates: 8
countries:
<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Po


Problems
Missing values in `discount`, `unit_price`, and `customer_id`
Duplicate rows 
 `country` has inconsistent capitalisation and extra spaces
 `order_date` is a string — needs parsing to datetime
Rows with no `customer_id` cannot be attributed to a customer and must be dropped

## 2. Cleaning

Удаление дубликатов:

In [3]:
print("Before:", df.shape)

df = df.drop_duplicates()

print("After:", df.shape)

Before: (208, 9)
After: (200, 9)


In [4]:
df["country"] = df["country"].str.strip().str.title()

Заполнение пропусков:
- `unit_price` → median медиан лучше чем mean вывод авг более четче

In [5]:
df["discount"] = df["discount"].fillna(0)

median_price = df["unit_price"].median()
df["unit_price"] = df["unit_price"].fillna(median_price)

Удаление строк без `customer_id`:

In [6]:
df = df.dropna(subset=["customer_id"])

Преобразование даты:

In [7]:
df["order_date"] = pd.to_datetime(df["order_date"])

Проверка нулевых пропусков:

In [8]:
print(df[["discount",
          "unit_price",
          "customer_id",
          "order_date"]].isnull().sum())

discount       0
unit_price     0
customer_id    0
order_date     0
dtype: int64


Должно показать:

```
discount       0
unit_price     0
customer_id    0
order_date     0
```

## 3. Enrich

Добавляем `revenue`:

In [9]:
df["revenue"] = (
    df["quantity"]
    * df["unit_price"]
    * (1 - df["discount"])
)

Добавляем месяц:

In [10]:
df["month"] = df["order_date"].dt.to_period("M")

Показать результат:

In [11]:
df[[
    "quantity",
    "unit_price",
    "discount",
    "revenue",
    "month"
]].head()

,quantity,unit_price,discount,revenue,month
1,3,59.99,0.10,161.973,2025-07
2,1,799.00,0.05,759.050,2025-02
3,4,899.00,0.20,2876.800,2025-12
4,5,549.00,0.10,2470.500,2025-08
5,6,59.99,0.15,305.949,2025-02


**Revenue formula:** `revenue = quantity × unit_price × (1 − discount)`

## 4. Merge

Проверяем `customer_id`:

In [12]:
print(customers["customer_id"].is_unique)

True


In [13]:
customers = customers.drop_duplicates(subset=["customer_id"])

In [14]:
before_rows = len(df)

Обязательно одинаковый тип:

In [15]:
df["customer_id"] = df["customer_id"].astype(int)
customers["customer_id"] = customers["customer_id"].astype(int)

Merge:

In [16]:
df = df.merge(
    customers,
    on="customer_id",
    how="left"
)

После merge:

In [17]:
after_rows = len(df)

print("Before:", before_rows)
print("After :", after_rows)

Before: 193
After : 193


## 5. Aggregations

**Revenue by Category**

In [18]:
category_summary = (
    df.groupby("category")
    .agg(
        total_revenue=("revenue", "sum")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

category_summary

,total_revenue
category,
Laptops,161187.4000
Phones,63403.4000
Monitors,58295.5500
Accessories,10323.5465


**Revenue by Month**

In [19]:
month_summary = (
    df.groupby("month")
    .agg(
        total_revenue=("revenue", "sum")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

month_summary

,total_revenue
month,
2025-07,42529.4260
2025-10,33697.7500
2025-08,30827.4315
2025-04,26456.2405
2025-06,24754.8375
2025-05,23633.5065
2025-12,22739.7510
2025-03,19836.5880
2025-02,19631.0780


**Revenue by Segment**

In [20]:
segment_summary = (
    df.groupby("segment")
    .agg(
        total_revenue=("revenue", "sum")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

segment_summary

,total_revenue
segment,
Consumer,174850.8365
Education,77782.0320
Business,40577.0280


Проверка для защиты — все три таблицы должны давать одинаковую общую сумму:

In [21]:
total = df["revenue"].sum()

top_cat       = category_summary.index[0]
top_cat_rev   = category_summary["total_revenue"].iloc[0]
top_cat_share = top_cat_rev / total * 100

best_month     = str(month_summary.index[0])
best_month_rev = month_summary["total_revenue"].iloc[0]

top_seg       = segment_summary.index[0]
top_seg_share = segment_summary["total_revenue"].iloc[0] / total * 100

low_cat       = category_summary.index[-1]
low_cat_share = category_summary["total_revenue"].iloc[-1] / total * 100

print(f"Total revenue : {total:>12,.2f}")
print(f"Top category  : {top_cat}  —  {top_cat_share:.1f}% of total")
print(f"Best month    : {best_month}  —  {best_month_rev:,.2f}")
print(f"Top segment   : {top_seg}  —  {top_seg_share:.1f}% of total")
print(f"Lowest cat    : {low_cat}  —  {low_cat_share:.1f}% of total")

Total revenue :   293,209.90
Top category  : Laptops  —  55.0% of total
Best month    : 2025-07  —  42,529.43
Top segment   : Consumer  —  59.6% of total
Lowest cat    : Accessories  —  3.5% of total


## 6. Conclusions

Вычисляем цифры для выводов:

In [22]:
print(category_summary["total_revenue"].sum())
print(month_summary["total_revenue"].sum())
print(segment_summary["total_revenue"].sum())

293209.8965
293209.8965
293209.8965


**Findings** *(all numbers come from the cell above)*

1. **Top category** — the highest-revenue category is printed above; it accounts for the share shown, indicating strong concentration in one product line.
2. **Best month** — the month with peak revenue is printed above; this likely reflects a seasonal spike or promotional campaign.
3. **Leading segment** — the top customer segment drives the share shown; it is the primary audience for retention efforts.
4. **Surprising observation** — the lowest-revenue category holds a very small share, suggesting it may be underperforming or newly launched.
5. **Data quality impact** — after cleaning, all targeted columns show zero missing values, validating the reliability of every aggregation above.